In [6]:
import cv2
import mediapipe as mp
from mediapipe import solutions
# removed unused import of landmark_pb2 (not needed in this cell)

# Correct MediaPipe solution access
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Open webcam stream
cap = cv2.VideoCapture(0)

# Configure Face Mesh
with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Webcam successfully opened! Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()

        if not success:
            print("Ignoring empty camera frame.")
            continue

        # Convert image to RGB
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process image
        results = face_mesh.process(image)

        # Convert back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw face mesh
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:

                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=
                    mp_drawing_styles.get_default_face_mesh_tesselation_style()
                )

        # Show window
        cv2.imshow('MediaPipe Face Mesh Test', image)

        # Exit on q
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# Release resources
cap.release()
cv2.destroyAllWindows()

Webcam successfully opened! Press 'q' to exit.


In [9]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

# Correct MediaPipe solution access
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Define landmark indices for Left and Right Eyes
LEFT_EYE_VERT_1 = [160, 144]  # Upper and lower eyelid points 1
LEFT_EYE_VERT_2 = [158, 153]  # Upper and lower eyelid points 2
LEFT_EYE_HORIZ  = [33, 133]   # Inner and outer corners

RIGHT_EYE_VERT_1 = [385, 380] # Upper and lower eyelid points 1
RIGHT_EYE_VERT_2 = [387, 373] # Upper and lower eyelid points 2
RIGHT_EYE_HORIZ  = [362, 263] # Inner and outer corners

# EAR Threshold to determine if eye is closed
EAR_THRESHOLD = 0.23  # If EAR goes below this, eye is considered closed

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    """
    Calculates the Eye Aspect Ratio (EAR) based on Euclidean distances of landmarks.
    """
    try:
        # Extract pixel coordinates for vertical landmarks
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        
        # Extract pixel coordinates for horizontal landmarks
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        # Calculate Euclidean distances
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        
        # Apply the EAR formula
        ear = (dist_v1 + dist_v2) / (2.0 * dist_h)
        return ear
    except Exception as e:
        return 0.0

# Open webcam stream
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("EAR Tracking successfully started! Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        
        # Convert image to RGB for MediaPipe processing
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        # Convert back to BGR for drawing
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # Calculate EAR for both eyes
                left_ear = calculate_ear(face_landmarks.landmark, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(face_landmarks.landmark, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                
                # Average EAR value
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Display EAR value on the video screen
                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                
                # Visual Alert Logic if EAR drops below threshold
                if avg_ear < EAR_THRESHOLD:
                    cv2.putText(image, "DROWSY ALERT!", (30, 100), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                
                # Draw only the eye contours to keep it clean and professional
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        # Show visualization window
        cv2.imshow('Driver Safety AI - EAR Tracking MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

EAR Tracking successfully started! Press 'q' to exit.


In [10]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

# Initialize MediaPipe modules
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Landmark indices for Left and Right Eyes
LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]

RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# ---- TUNED DROWSINESS CONFIGURATION ----
EAR_THRESHOLD = 0.21          # Tuned based on your squinting threshold
CONSECUTIVE_FRAMES_LIMIT = 15 # Eye must be closed for at least 15 consecutive frames (~0.5 - 1 second) to alert
FRAME_COUNTER = 0             # Keeps track of consecutive closed-eye frames

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    """Calculates the Eye Aspect Ratio (EAR) based on Euclidean distances."""
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

# Open webcam stream
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Advanced Time-Series EAR Tracking Started... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                left_ear = calculate_ear(face_landmarks.landmark, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(face_landmarks.landmark, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Check if EAR is below threshold
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1 # Increment frame count if eyes are closed
                else:
                    FRAME_COUNTER = 0 # RESET counter instantly if eyes are opened even for a single frame

                # Display real-time EAR and active frame buffer metrics
                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                cv2.putText(image, f"Closed Frames: {FRAME_COUNTER}", (30, 90), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
                
                # Trigger alert ONLY if the eyes are closed continuously beyond the frame limit
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "!!! DROWSY ALERT !!!", (30, 140), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
                
                # Draw facial contours
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Advanced EAR MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Advanced Time-Series EAR Tracking Started... Press 'q' to exit.
